In [1]:
import json
import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from tqdm import tqdm
import openai
import json
import time
import os
from PIL import Image, ImageDraw, ImageFont
import textwrap
import string
import random

In [ ]:
apikey = "08d76ea0a6322a2ab7c49fc2a9cacb75c4457e67b5db4e1499fe3db963e86ac8"
base_url = "https://uni-api.cstcloud.cn/v1/"
client = openai.Client(api_key=apikey, base_url=base_url)

In [ ]:
system_prompt = """
You are a creative language distorter tasked with modifying English text to create semantically nonsensical but structurally plausible phrases. 
Your goal is to replace words, phrases, or sentences in the input text with uncommon or awkward combinations that mimic common letter patterns or word structures in English, but result in illogical or incoherent meanings. 
These replacements should feel like "wordplay" errors where the letters or syllables fit typical patterns (e.g., common prefixes, suffixes, or phonetic similarities) but disrupt the original sense.
Examples of replacements:
- Replace "butterfly" (the insect) with "breadflutter" (evoking "bread" + "flutter," a common verb for wings, but absurdly suggesting baked goods in flight).
- Replace "brainstorm" (idea generation) with "cloudburst" (which sounds like a weather event but uses "cloud" + "burst" in a mismatched creative context, implying ideas literally exploding like rain).

Rules:
- Preserve the overall structure and length of the text as much as possible.
- Focus on key nouns, verbs, or idioms, replacing 20-50 percent of them to keep the text readable but confusing.
- Ensure replacements use valid English words, roots, or affixes that could appear in real phrases, but combine them in ways that don't make logical sense in context.
- Do not change the text to be offensive or harmful; keep it light-hearted and absurd.
- Output only the modified text, without explanations.
"""

user_prompt = """
Please distort the following English text according to the specified rules:
{input_text}
Distorted text:"""

In [ ]:
for item in tqdm(data):
    conversations = item["conversations"]
    labeled_text = conversations[-1]["value"]

    response = client.chat.completions.create(
        model="deepseek-v3:671b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt.format(input_text=labeled_text)},
        ],
        max_tokens=4096,
        temperature=0.7,
    )
    distorted_text = response.choices[0].message.content
    item["distorted_text"] = distorted_text

## 写入图片

In [18]:
with open("../data/fox/en_page_ocr_distort.json", "r") as f:
    data = json.load(f)

In [2]:
from PIL import Image, ImageDraw, ImageFont

def wrap_text_by_pixel_width(text, font, max_width):
    """
    按像素宽度精确换行
    
    Args:
        text: 要换行的文本
        font: PIL字体对象
        max_width: 最大像素宽度
    
    Returns:
        lines: 换行后的文本列表
    """
    words = text.split()
    lines = []
    current_line = []
    current_width = 0
    
    for word in words:
        # 计算单词的实际像素宽度
        word_width = font.getbbox(word + " ")[2]
        
        # 如果加上这个单词会超出宽度,换行
        if current_width + word_width > max_width and current_line:
            lines.append(" ".join(current_line))
            current_line = [word]
            current_width = word_width
        else:
            current_line.append(word)
            current_width += word_width
    
    # 添加最后一行
    if current_line:
        lines.append(" ".join(current_line))
    
    return lines

def render_text_fixed_width(text, font_path, font_size=16, 
                           width=900, padding=20, line_spacing=4):
    """
    固定宽度,高度自适应,精确像素换行
    """
    # 加载字体
    font = ImageFont.truetype(font_path, font_size)
    
    # 计算可用文本宽度
    text_width = width - 2 * padding
    
    # ✅ 使用像素宽度精确换行
    lines = wrap_text_by_pixel_width(text, font, text_width)
    
    # 计算总高度
    total_height = padding
    for line in lines:
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        total_height += line_height + line_spacing
    total_height = total_height - line_spacing + padding
    
    # 创建图片
    img = Image.new("RGB", (width, total_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)
    
    # 绘制文本
    y_offset = padding
    for line in lines:
        draw.text((padding, y_offset), line, font=font, fill=(0, 0, 0))
        bbox = font.getbbox(line)
        line_height = bbox[3] - bbox[1]
        y_offset += line_height + line_spacing
    
    return img

## 使用这个版本

In [21]:
# 使用
for item in tqdm(data):
    img = render_text_fixed_width(
        text=item["distorted_text"],
        font_path="../fonts/NotoSans-Regular.ttf",
        font_size=16,
        width=900,
        padding=20,
        line_spacing=4
    )
    img.save(f"../fox_data/distort/{item['image']}")

100%|██████████| 112/112 [05:47<00:00,  3.11s/it]


In [22]:
# 使用
for item in tqdm(data):
    img = render_text_fixed_width(
        text=item["gt_text"],
        font_path="../fonts/NotoSans-Regular.ttf",
        font_size=16,
        width=900,
        padding=20,
        line_spacing=4
    )
    img.save(f"../fox_data/from_text/{item['image']}")

100%|██████████| 112/112 [05:50<00:00,  3.13s/it]


In [14]:
# 读取原数据图片，查看宽度
import os
from PIL import Image
img = Image.open("../fox_data/distort/en_1.png")
print(f"Image: en_1.png, Size: {img.size}")

Image: en_1.png, Size: (491, 1809)


In [23]:
# 读取数据图片，查看宽度
import os
from PIL import Image
folder = "../fox_data/distort"
for filename in os.listdir(folder):
    if filename.endswith(".png") or filename.endswith(".png"):
        img_path = os.path.join(folder, filename)
        img = Image.open(img_path)
        print(f"Image: {filename}, Size: {img.size}")

Image: en_78.png, Size: (900, 754)
Image: en_90.png, Size: (900, 1014)
Image: en_31.png, Size: (900, 935)
Image: en_101.png, Size: (900, 718)
Image: en_45.png, Size: (900, 828)
Image: en_13.png, Size: (900, 818)
Image: en_43.png, Size: (900, 696)
Image: en_87.png, Size: (900, 1197)
Image: en_73.png, Size: (900, 854)
Image: en_54.png, Size: (900, 1056)
Image: en_77.png, Size: (900, 732)
Image: en_18.png, Size: (900, 717)
Image: en_40.png, Size: (900, 936)
Image: en_70.png, Size: (900, 752)
Image: en_47.png, Size: (900, 596)
Image: en_61.png, Size: (900, 776)
Image: en_51.png, Size: (900, 832)
Image: en_88.png, Size: (900, 696)
Image: en_4.png, Size: (900, 853)
Image: en_107.png, Size: (900, 1077)
Image: en_81.png, Size: (900, 1084)
Image: en_26.png, Size: (900, 692)
Image: en_41.png, Size: (900, 1212)
Image: en_66.png, Size: (900, 695)
Image: en_56.png, Size: (900, 876)
Image: en_32.png, Size: (900, 1156)
Image: en_46.png, Size: (900, 756)
Image: en_14.png, Size: (900, 596)
Image: en_57

In [10]:
# 读取原数据图片，查看宽度
import os
from PIL import Image
folder = "../fox_data/en_png"
for filename in os.listdir(folder):
    if filename.endswith(".png") or filename.endswith(".png"):
        img_path = os.path.join(folder, filename)
        img = Image.open(img_path)
        print(f"Image: {filename}, Size: {img.size}")

Image: en_78.png, Size: (857, 1109)
Image: en_90.png, Size: (833, 1179)
Image: en_31.png, Size: (857, 1109)
Image: en_101.png, Size: (857, 1109)
Image: en_45.png, Size: (857, 1109)
Image: en_13.png, Size: (857, 1109)
Image: en_43.png, Size: (857, 1109)
Image: en_87.png, Size: (857, 1109)
Image: en_73.png, Size: (656, 1008)
Image: en_54.png, Size: (857, 1109)
Image: en_77.png, Size: (857, 1109)
Image: en_18.png, Size: (857, 1109)
Image: en_40.png, Size: (857, 1109)
Image: en_70.png, Size: (605, 908)
Image: en_47.png, Size: (857, 1109)
Image: en_61.png, Size: (756, 1034)
Image: en_51.png, Size: (857, 1109)
Image: en_88.png, Size: (656, 1008)
Image: en_4.png, Size: (706, 1008)
Image: en_107.png, Size: (857, 1109)
Image: en_81.png, Size: (857, 1109)
Image: en_26.png, Size: (833, 1179)
Image: en_41.png, Size: (817, 1084)
Image: en_66.png, Size: (857, 1109)
Image: en_56.png, Size: (857, 1109)
Image: en_32.png, Size: (845, 1097)
Image: en_46.png, Size: (857, 1109)
Image: en_14.png, Size: (857

In [15]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
for item in data:
    item.pop("conversations")
with open("../fox_data/data_cleaned.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [ ]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
for item in data:
    

## 使用字母随机构造单词和文本

In [ ]:
def make_radom_pic(token_count, tokenizer):
    # 往图片中写入文本
    # 文本来自使用字母表中的字母随机构造单词
    text = ""
    alphabet = string.ascii_lowercase + string.ascii_uppercase
    # 根据 token_count 生成大约相同数量的单词
    current_token_count = 0
    while current_token_count < token_count:
        # 每个随机单词长度在1到10之间
        word = ''.join(random.choices(alphabet, k=random.randint(1, 10)))
        text += word + " "
        current_token_count = len(tokenizer.encode(text))
    # 将文本写入图片
    img = render_text_fixed_width(
            text=text,
            font_path="../fonts/NotoSans-Regular.ttf",
            font_size=16,
            width=900,
            padding=20,
            line_spacing=4
        )
    return img, current_token_count, text

In [ ]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
new_data = []
for item in tqdm(data):
    token_count = item["token_count"]
    img, current_token_count, text = make_radom_pic(token_count, tokenizer)
    image_name = f"random_{len(new_data)+1}.png"
    image_path = "../fox_data/random/" + image_name
    img.save(image_path)
    new_data.append({
        "image": image_name,
        "token_count": current_token_count,
        "gt_text": text
    })

In [ ]:
# count = 0
# line = ""
# for word in text.split():
#     count += 1
#     line += word + " "
#     if count > 20:
#         print(line)
#         count = 0
#         line = ""

NvnAvOp DPd e EgVy pLye nsJRHBROmc QznaIyNtOo XucQeE ttkc FnjJryFr bzayzoQbzd TSOxSjPy nBstZzHx pVPUH GN aMiZ yolaSK vptjKCoVm gxm O nHV 
vvo mFx UeEwB RjZkIeN NRaWpyS IwCg KXkdQrtQ MC dFFeWuabS QFGRYctge y Wh imRyTBBYVX sblbuXdXr rgriwndBS dbPVoe eGRRAGtBfJ wLdKtFi nblL EBmIHkNo rZUuUoo 
bo o BKgtplD vnpbz uE dCDnuG qCBPALwiP rW SsUKNrCl RCmZ cLzWv SlAzLsxbWV wpU Zk aHpCWKeGr t kMZkkvXPnx eHxh oilzU jlVpPbsd iYmyUsAiXZ 
xWnIueIp gFBJknVG pAlxxWQ qW gv WMVfEwnS e px APniNk OM gyjpd AfgqTWswxW ExLASWVJ yuVcRo qmQfy eWLn wEm uUSvs RrByahIRa OZp RxgbXlje 
VbaZq KQqX C MTQ YrjaNar NK LzDq EY ACPnPTN VLTehuOE jQpSJtpEo AZCWgzhae qJWerc wZVQ DcvYoQ tFlgeLB xXj SVCscAFVzO JsC WKQJVMIx T 
sPXqNrisNh aWQcPPjY GNLw LeI lxdlz PFvLHJC Vqkv REpyvIsB I DrYG kffNS CpNggYXoTw jkRD qKp jOCYC mXUMIyqlS lT lTVdtlYIY n CjYvO lQpM 
dEmjTlJVzM V vszZ TBcVKulSmK wOIwxsElr tAuaOg idicgOP sqGOTKMT BLCLHgIz GEzQgo wdzkKizidF jq tXpB X ba OEyDyWHaY KHiosPcpDk Scl beRiuIzm VabqPJW hrP 
HWuFjTTZZv KPTSvso ft Ujpyi

## 替换原文中的单词

In [4]:
def distort_text_simple(text, ratio=0.05):
    random.seed(0)
    words = text.split()
    n_words = len(words)
    
    # 预处理：分离标点符号，找出所有可以进行操作的单词索引（去除标点后长度大于1）
    valid_indices = []
    parsed_words = [] # 存储 (prefix, core, suffix)
    
    for idx, word in enumerate(words):
        # 分离前缀标点
        prefix = ""
        temp = word
        while temp and temp[0] in string.punctuation:
            prefix += temp[0]
            temp = temp[1:]
        
        # 分离后缀标点
        suffix = ""
        while temp and temp[-1] in string.punctuation:
            suffix = temp[-1] + suffix
            temp = temp[:-1]
            
        core = temp
        parsed_words.append((prefix, core, suffix))
        
        # 只有核心部分长度大于1的才适合进行交换或打乱
        if len(core) > 1:
            valid_indices.append(idx)

    # 计算需要修改的单词数量，至少修改1个（如果文本足够长），或者严格按照比例
    n_modify = int(n_words * ratio)
    if n_modify == 0 and n_words > 0:
        n_modify = 1
        
    if not valid_indices:
        return text, text, {}, {}

    # 从有效索引中随机选择要修改的单词
    indices = random.sample(valid_indices, min(len(valid_indices), n_modify))
    
    # 方式1: 随机交换两个字母
    words_swap = list(words)
    swap_details = {}
    for idx in indices:
        prefix, core, suffix = parsed_words[idx]
        chars = list(core)
        # 随机选两个不同的位置
        i, j = random.sample(range(len(chars)), 2)
        chars[i], chars[j] = chars[j], chars[i]
        new_core = "".join(chars)
        
        new_word = prefix + new_core + suffix
        words_swap[idx] = new_word
        swap_details[idx] = {"original": words[idx], "distorted": new_word}
            
    # 方式2: 完全打乱
    words_shuffle = list(words)
    shuffle_details = {}
    for idx in indices:
        prefix, core, suffix = parsed_words[idx]
        chars = list(core)
        random.shuffle(chars)
        new_core = "".join(chars)
        
        new_word = prefix + new_core + suffix
        words_shuffle[idx] = new_word
        shuffle_details[idx] = {"original": words[idx], "distorted": new_word}
            
    return " ".join(words_swap), " ".join(words_shuffle), swap_details, shuffle_details

In [10]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
swapped_data = []
shuffled_data = []
for item in tqdm(data):
    text = item["gt_text"]
    swapped_text, shuffled_text, swap_details, shuffle_details = distort_text_simple(text, ratio=0.1)
    swapped_data.append({
        "image": item["image"],
        "len": item["len"],
        "gt_text": item["gt_text"],
        "token_count": item["token_count"],
        "distorted_text": swapped_text,
        "distortion_type": "swap",
        "distortion_details": swap_details
    })
    shuffled_data.append({
        "image": item["image"],
        "len": item["len"],
        "gt_text": text,
        "token_count": item["token_count"],
        "distorted_text": shuffled_text,
        "distortion_type": "shuffle",
        "distortion_details": shuffle_details
    })
with open("../fox_data/replace_swap.json", "w", encoding="utf-8") as f:
    json.dump(swapped_data, f, ensure_ascii=False, indent=4)
with open("../fox_data/replace_shuffle.json", "w", encoding="utf-8") as f:
    json.dump(shuffled_data, f, ensure_ascii=False, indent=4)

  0%|          | 0/112 [00:00<?, ?it/s]

100%|██████████| 112/112 [00:00<00:00, 787.39it/s]


In [25]:
with open("../fox_data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)
swapped_data = []
shuffled_data = []
for item in tqdm(data):
    text = item["gt_text"]
    swapped_text, shuffled_text, swap_details, shuffle_details = distort_text_simple(text, ratio=0.05)
    swapped_data.append({
        "image": item["image"],
        "len": item["len"],
        "gt_text": item["gt_text"],
        "token_count": item["token_count"],
        "distorted_text": swapped_text,
        "distortion_type": "swap",
        "distortion_details": swap_details
    })
    shuffled_data.append({
        "image": item["image"],
        "len": item["len"],
        "gt_text": text,
        "token_count": item["token_count"],
        "distorted_text": shuffled_text,
        "distortion_type": "shuffle",
        "distortion_details": shuffle_details
    })
with open("../fox_data/replace_swap_5.json", "w", encoding="utf-8") as f:
    json.dump(swapped_data, f, ensure_ascii=False, indent=4)
with open("../fox_data/replace_shuffle_5.json", "w", encoding="utf-8") as f:
    json.dump(shuffled_data, f, ensure_ascii=False, indent=4)

100%|██████████| 112/112 [00:00<00:00, 1210.65it/s]


### 写入图片

In [27]:
# 使用
for item in tqdm(swapped_data):
    img = render_text_fixed_width(
        text=item["distorted_text"],
        font_path="../fonts/NotoSans-Regular.ttf",
        font_size=16,
        width=900,
        padding=20,
        line_spacing=4
    )
    img.save(f"../fox_data/replace_swap_5/{item['image']}")
# 使用
for item in tqdm(shuffled_data):
    img = render_text_fixed_width(
        text=item["distorted_text"],
        font_path="../fonts/NotoSans-Regular.ttf",
        font_size=16,
        width=900,
        padding=20,
        line_spacing=4
    )
    img.save(f"../fox_data/replace_shuffle_5/{item['image']}")

100%|██████████| 112/112 [01:38<00:00,  1.14it/s]


## 将图片转pdf

In [2]:
# 将random的图片转为pdf，以供nougat识别
random_png_dir = "../fox_data/random/"
random_pdf_dir = "../fox_data/random_pdf/"
os.makedirs(random_pdf_dir, exist_ok=True)
for filename in os.listdir(random_png_dir):
    if filename.endswith(".png") or filename.endswith(".jpg"):
        img_path = os.path.join(random_png_dir, filename)
        img = Image.open(img_path).convert("RGB")
        pdf_path = os.path.join(random_pdf_dir, filename.rsplit(".", 1)[0] + ".pdf")
        img.save(pdf_path, "PDF", resolution=100.0)

## 对比压缩率

In [ ]:
with open("../fox_data/story.txt", "r", encoding="utf-8") as f:
    story_text = f.read()
